# **Interactive visualization of the ACE cruise route and CTD bottle samples**

*Sampling locations and near-surface hydrographic and biogeochemical observations from the Antarctic Circumnavigation Expedition*

For an interactive version of this page please visit the Google Colab: [Open in Google Colab](https://colab.research.google.com/drive/1iVjpHV-2t8A8b_n3yBo0IBwtrUWZQ8vB#scrollTo=CQqjXzCQIX_I)

*(To open link in new tab press Ctrl + click)*

Alternatevly this notebook can be opened with Binder by following the link: [Interactive visualization of the ACE cruise route and CTD bottle samples](https://mybinder.org/v2/gh/s4oceanice/literacy.s4oceanice/main?urlpath=%2Fdoc%2Ftree%2Fnotebooks_binder%2FACE_route_and_samples.ipynb)

**Scientific background**

Oceanographic expeditions provide essential observations for understanding the physical and biogeochemical properties of the Southern Ocean.

During CTD casts, instruments measure the vertical structure of the water column, while bottles mounted on the rosette collect discrete water samples at selected depths. These observations can be used to investigate spatial variability in temperature, salinity, density, dissolved oxygen, light availability, and other environmental properties.

Mapping the sampling stations together with the measured values provides an immediate overview of the cruise coverage and helps identify spatial gradients along the expedition route.

**Cruise overview**

This notebook retrieves ACE CTD bottle observations from the OCEAN:ICE ERDDAP service. It displays the sampling locations on an interactive map and allows users to colour the markers according to selected hydrographic and biogeochemical variables.

**Notebook objectives**

This notebook enables users to:

- retrieve ACE CTD bottle data from ERDDAP;
- remove quality-control flag columns from the visualization choices;
- identify the shallowest available record for each bottle grouping;
- visualize the spatial distribution of selected parameters;
- examine the approximate sampling sequence along the cruise route.


**Dataset description**

The notebook accesses the ERDDAP table dataset:

https://er1.s4oceanice.eu/erddap/tabledap/ACE_bottle_CTD20200406CURRSGCMR

The query retrieves observations collected between **9 March 2017** and **16 March 2017**.

The available information includes:

- expedition, section, station, cast, sample, and bottle identifiers;
- observation time;
- latitude and longitude;
- pressure and depth;
- CTD temperature;
- salinity;
- density;
- sound velocity;
- dissolved oxygen and oxygen saturation;
- fluorescence, backscatter, and photosynthetically active radiation;
- quality-control flags associated with several measured variables.

The quality-control columns are excluded from the interactive parameter selector, while the original measurement columns are retained.

**How to use this notebook**

1. Run the code cells sequentially from top to bottom.
2. Wait for the remote dataset to be downloaded and processed.
3. Use the available menus, sliders, or map controls to select the variables and periods of interest.
4. Read the interpretation guidance before drawing scientific conclusions from the visualizations.

The notebook retrieves data from remote services. An active internet connection is therefore required.


**Data retrieval**

The following cell defines the ERDDAP query and downloads the source table. ERDDAP returns a units row below the column headers; this row is removed in the preparation stage.


In [ ]:
# @title
import pandas as pd

# dataset URL
DATA_URL = 'https://er1.s4oceanice.eu/erddap/tabledap/ACE_bottle_CTD20200406CURRSGCMR.csv?EXPOCODE%2CSECT_ID%2CSTNNBR%2CCASTNO%2Ctime%2Clatitude%2Clongitude%2CSAMPNO%2CBTLNBR%2CBTLNBR_FLAG_W%2CPRES%2CCTDPRS_FLAG_W%2Cdepth%2CCTDTMP%2CCTDTMP_FLAG_W%2CCTDSAL%2CCTDSAL_FLAG_W%2CCTDDENS%2CCTDDENS_FLAG_W%2CCTDSOUND%2CCTDSOUND_FLAG_W%2CCTDOXY%2CCTDOXY_FLAG_W%2CCTDOXYSAT%2CCTDOXYSAT_FLAG_W%2CCTDFLUOR1%2CCTDFLUOR1_FLAG_W%2CCTDFLUOR2%2CCTDFLUOR2_FLAG_W%2CCTDFLUOR2Q%2CCTDFLUOR2Q_FLAG_W%2CBACKSC%2CBACKSC_FLAG_W%2CPAR%2CPAR_FLAG_W&time%3E=2017-03-09T00%3A00%3A00Z&time%3C=2017-03-16T10%3A54%3A03Z'

df = pd.read_csv(DATA_URL)

#display(df.head())
#print(df.info())

**Data preparation and interactive visualization**

The following section removes the units row and quality-control fields, converts the principal measurement columns to numeric values, excludes invalid coordinates, groups the observations by expedition and sampling identifiers, and retains the shallowest available record for each bottle grouping.

The cleaned records are then displayed using an interactive `folium` map and a parameter dropdown.



**Interactive visualization**

Use the dropdown menu to select the parameter represented by the marker colours.

- Marker positions represent bottle-sampling coordinates.
- Marker colours represent the selected numerical variable.
- The colour bar shows the displayed value range.
- Popups report the selected value and observation time.
- The grey line provides an approximate representation of the sampling sequence.

In [ ]:
# @title
import folium
import branca.colormap as cm
from ipywidgets import interact, Dropdown
import numpy as np
import pandas as pd

df_clean = df.drop(index=0).copy()

cols_to_keep_initial = [c for c in df_clean.columns if not c.endswith('_FLAG_W')]
df_clean = df_clean[cols_to_keep_initial]

for col in ['latitude', 'longitude', 'depth', 'CTDTMP', 'CTDSAL', 'CTDDENS', 'CTDOXY', 'CTDOXYSAT', 'PAR', 'STNNBR', 'CASTNO', 'BTLNBR']:
    if col in df_clean.columns:
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

df_clean = df_clean.dropna(subset=['latitude', 'longitude'])

grouping_columns = ['EXPOCODE', 'SECT_ID', 'STNNBR', 'CASTNO', 'BTLNBR']

df_clean = df_clean.dropna(subset=grouping_columns + ['depth'])

df_clean = df_clean.loc[df_clean.groupby(grouping_columns)['depth'].idxmin()]

map_params = df_clean.select_dtypes(include=[np.number]).columns.tolist()

map_params = [p for p in map_params if p not in ['latitude', 'longitude', 'STNNBR', 'CASTNO', 'SAMPNO', 'BTLNBR']]

def plot_map(parameter):
    map_center = [df_clean['latitude'].mean(), df_clean['longitude'].mean()]
    m = folium.Map(location=map_center, zoom_start=4, control_scale=True)

    vmin, vmax = df_clean[parameter].min(), df_clean[parameter].max()
    if vmin == vmax:
        vmax = vmin + 1
    colormap = cm.LinearColormap(colors=['blue', 'green', 'yellow', 'red'], vmin=vmin, vmax=vmax)
    colormap.caption = f'Valore di {parameter}'

    for _, row in df_clean.iterrows():
        if not np.isnan(row[parameter]):
            folium.CircleMarker(
                location=[row['latitude'], row['longitude']],
                radius=5,
                color=colormap(row[parameter]),
                fill=True,
                fill_color=colormap(row[parameter]),
                fill_opacity=0.7,
                popup=f"{parameter}: {row[parameter]:.2f} @ {row['time']}"
            ).add_to(m)

    points = df_clean[['latitude', 'longitude']].values.tolist()
    folium.PolyLine(points, color="gray", weight=1, opacity=0.5).add_to(m)

    colormap.add_to(m)
    return m

interact(plot_map, parameter=Dropdown(options=map_params, value='CTDTMP'));

interactive(children=(Dropdown(description='parameter', index=1, options=('depth', 'CTDTMP', 'CTDSAL', 'CTDDEN…

**Interpretation guidance**

The visualization is intended for exploratory analysis. The retained observations correspond to the shallowest available record within each expedition, section, station, cast, and bottle grouping and do not represent complete vertical profiles.

Spatial differences may reflect environmental gradients, but they may also be affected by sampling time, station spacing, bottle depth, missing values, and variable-specific measurement coverage. The connecting line follows the order of the filtered records and should not be interpreted as the exact vessel trajectory.

**Additional resources and acknowledgement**

Main resources used across the notebook series include:

- [pandas](https://pandas.pydata.org/docs/) for data retrieval and preparation
- [numpy](https://numpy.org/doc/) for numerical operations
- [Folium](https://python-visualization.github.io/folium/) for interactive mapping
- [Matplotlib](https://matplotlib.org/stable/)
- [branca](https://python-visualization.github.io/branca/) for the continuous colour scale
- [ipywidgets](https://ipywidgets.readthedocs.io/) for the parameter dropdown.

The data are made available through the OCEAN:ICE ERDDAP infrastructure.

This work has received funding from the European Union Horizon Europe project **Ocean-Cryosphere Exchanges in ANtarctica: Impacts on Climate and the Earth System (OCEAN:ICE)** under Grant Agreement No. 101060452. UK partners are funded by UK Research and Innovation under the UK Government's Horizon Europe funding guarantee.


<center>
  <div style="display: flex; justify-content: center; align-items: flex-start; gap: 80px;">
    <img src="https://ocean-ice.eu/wp-content/uploads/2025/02/TO-USE-RGB-for-digital-materials-V.png" height="140" style="margin-top: 50px;"/>
    <img src="https://ocean-ice.eu/wp-content/uploads/2025/02/UKRI-logo-1.png" height="100"/>
    <img src="https://ocean-ice.eu/wp-content/uploads/2023/06/logo-polar-cluster-2.png" height="100"/>
  </div>
</center>